# Object Storage — teste no Medallion_Compute
Adaptado de [Access_Object_Storage_Data.ipynb](https://github.com/oracle-samples/oracle-aidp-samples/blob/main/getting-started/Access_Object_Storage_Data.ipynb).

Oracle AI Data Platform v1.0 — Copyright © 2025, Oracle and/or its affiliates.
Licensed under the Universal Permissive License v1.0: https://oss.oracle.com/licenses/upl/

Testa leitura/escrita direta em Object Storage, volume externo e tabela externa CSV.
Execute no AIDP. O parâmetro OCI_BUCKET é obrigatório (`oci://bucket@namespace`).
Cada execução usa prefixo e nomes únicos; somente dados sintéticos são gravados.
O catálogo de destino deve ser interno (padrão `default`), não o catálogo remoto Oracle ALH.
Volume e tabela externa são provisionados pela API AIDP e reutilizados; os arquivos permanecem para inspeção.

In [ ]:
import json
import re
from uuid import uuid4
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com um compute anexado.')

def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, default)
    return default if value is None else str(value).strip()

def identifier(value):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        raise ValueError(f'Identificador inválido: {value!r}')
    return f'`{value}`'

bucket = parameter('OCI_BUCKET', '').rstrip('/')
if not re.fullmatch(r'oci://[A-Za-z0-9_.-]+@[A-Za-z0-9_-]+', bucket):
    raise ValueError('OCI_BUCKET deve ser o bucket de teste: oci://bucket@namespace, sem prefixo.')
catalog = parameter('CATALOG_NAME', 'default')
schema = parameter('SCHEMA_NAME', 'default')
base_prefix = parameter('BASE_PREFIX', 'aidp_codex_tests/object_storage').strip('/')
if not re.fullmatch(r'[A-Za-z0-9_-]+(?:/[A-Za-z0-9_-]+)*', base_prefix):
    raise ValueError('BASE_PREFIX aceita somente diretórios simples, sem pontos ou caracteres SQL.')
run_id = uuid4().hex[:12]
root = f'{bucket}/{base_prefix}/{run_id}'
volume = parameter('EXTERNAL_VOLUME_NAME', 'aidp_os_test_volume')
volume_location = parameter('EXTERNAL_VOLUME_LOCATION', bucket + '/' + base_prefix + '/').rstrip('/') + '/'
table = parameter('EXTERNAL_TABLE_NAME', 'aidp_os_test_table')
table_location = parameter('EXTERNAL_TABLE_LOCATION', bucket + '/aidp_codex_tests/object_storage_tables/csv_test')
volume_id = '.'.join(identifier(x) for x in (catalog, schema, volume))
table_id = '.'.join(identifier(x) for x in (catalog, schema, table))
# Valida catálogo/schema e falha antes de gravar caso não estejam disponíveis.
spark.sql(f'SHOW TABLES IN {identifier(catalog)}.{identifier(schema)}').limit(1).collect()
result = {'run_id': run_id, 'object_storage_root': root, 'catalog': catalog, 'schema': schema}
print(json.dumps(result))

In [ ]:
# Dados sintéticos com tipos explícitos e comparação de valores, incluindo duplicatas.
data_schema = StructType([StructField('name', StringType()), StructField('age', IntegerType())])
records = [(f'Alice_{run_id}', 30), (f'Bob_{run_id}', 35), (f'Charlie_{run_id}', 25)]
df = spark.createDataFrame(records, data_schema)

def read_csv(path):
    return spark.read.schema(data_schema).option('header', True).option('mode', 'FAILFAST').csv(path)

def verify(frame, expected, label):
    actual = sorted((r['name'], int(r['age'])) for r in frame.collect())
    if actual != sorted(expected):
        raise AssertionError(f'{label}: valores lidos diferem dos esperados: {actual!r}')
    result[label] = {'status': 'SUCCESS', 'rows_verified': len(actual)}

## 1. Leitura e escrita direta em Object Storage

In [ ]:
direct_path = root + '/direct_csv'
df.write.mode('errorifexists').option('header', True).csv(direct_path)
verify(read_csv(direct_path), records, 'direct_object_storage')

## 2. Volume externo restrito ao prefixo desta execução

In [ ]:
# Volume provisionado pela API AIDP; o runtime atual não aceita CREATE EXTERNAL VOLUME.
volume_path = f'/Volumes/{catalog}/{schema}/{volume}/{run_id}/volume_csv'
df.write.mode('errorifexists').option('header', True).csv(volume_path)
verify(read_csv(volume_path), records, 'external_volume')
# Confere também o mesmo arquivo pela URI OCI de origem.
verify(read_csv(volume_location + run_id + '/volume_csv'), records, 'volume_underlying_objects')
result['external_volume'] = dict(result['external_volume'], name=f'{catalog}.{schema}.{volume}', path=volume_path)

## 3. Tabela externa CSV: consulta e atualização dos arquivos de origem

In [ ]:
# Tabela externa provisionada pela API do catálogo AIDP.
# Ela aponta para um diretório exclusivo de testes, compartilhado entre execuções.
from pyspark.sql import functions as F
table_path = table_location
names = [r[0] for r in records]
df.write.mode('append').option('header', True).csv(table_path)
spark.catalog.refreshTable(table_id)
verify(spark.table(table_id).filter(F.col('name').isin(names)), records, 'external_table_initial')
extra = [(f'Diana_{run_id}', 28)]
spark.createDataFrame(extra, data_schema).write.mode('append').option('header', True).csv(table_path)
spark.catalog.refreshTable(table_id)
verify(spark.table(table_id).filter(F.col('name').isin(names + [extra[0][0]])), records + extra, 'external_table_after_append')
result['external_table_name'] = f'{catalog}.{schema}.{table}'
result['external_table_location'] = table_path


## Evidência e retenção
O resultado deve conter SUCCESS em todas as verificações. Permanecem os objetos sob o prefixo único acima, um volume externo e uma tabela externa reutilizáveis.
Nenhum objeto de uma execução anterior é sobrescrito ou apagado. Limpeza posterior deve usar os caminhos/nomes emitidos no resultado.

In [ ]:
result['status'] = 'SUCCESS'
print('AIDP_OBJECT_STORAGE_TEST_RESULT=' + json.dumps(result, sort_keys=True))